# Regresja logistyczna — master notebook v7

Ten notebook jest wersją kodowaną do pliku `logistic_regression_master_lesson_v7.md`. Nie używa `statsmodels`; wszystkie kluczowe rachunki są robione ręcznie albo przez `numpy`, `scipy` i opcjonalnie `sklearn`.

## 1. Probability, odds i log-odds

$$odds=\frac{p}{1-p}$$

$$logit(p)=\log\left(\frac{p}{1-p}\right)$$

Pokaż studentom, że $p=0.5$ daje odds $1$ i log-odds $0$.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.special import expit
from scipy.stats import fisher_exact, chi2_contingency, chi2, norm, hypergeom

pd.set_option('display.precision', 6)

p = np.array([0.1, 0.25, 0.5, 0.75, 0.9])
odds = p / (1 - p)
log_odds = np.log(odds)
pd.DataFrame({'p': p, 'odds': odds, 'log_odds': log_odds})

## 2. Odds ratio, log odds ratio i testy

Tabela z przykładu mutacja genu / rak:

| grupa | Rak + | Rak - |
|---|---:|---:|
| Mutacja + | 23 | 117 |
| Mutacja - | 6 | 210 |

$$OR=\frac{ad}{bc}$$

$$SE(\log OR)=\sqrt{\frac1a+\frac1b+\frac1c+\frac1d}$$

In [ ]:
a, b, c, d = 23, 117, 6, 210
obs = np.array([[a, b], [c, d]])

or_hat = (a * d) / (b * c)
log_or = np.log(or_hat)
se_log_or = np.sqrt(1/a + 1/b + 1/c + 1/d)
z = log_or / se_log_or
p_wald = 2 * norm.sf(abs(z))
ci_log = (log_or - 1.96 * se_log_or, log_or + 1.96 * se_log_or)
ci_or = tuple(np.exp(ci_log))

row_sums = obs.sum(axis=1, keepdims=True)
col_sums = obs.sum(axis=0, keepdims=True)
expected = row_sums @ col_sums / obs.sum()
chi2_stat = ((obs - expected) ** 2 / expected).sum()
p_chi = chi2.sf(chi2_stat, df=1)

results = {
    'OR': or_hat,
    'log_OR': log_or,
    'SE_log_OR': se_log_or,
    'z_Wald': z,
    'p_Wald': p_wald,
    'CI_OR_low': ci_or[0],
    'CI_OR_high': ci_or[1],
    'chi2': chi2_stat,
    'p_chi2': p_chi,
    'p_Fisher_two_sided': fisher_exact(obs, alternative='two-sided').pvalue,
}
pd.Series(results)

In [ ]:
expected_df = pd.DataFrame(expected, index=['Mutacja +', 'Mutacja -'], columns=['Rak +', 'Rak -'])
expected_df

### Fisher exact test ręcznie

Przy ustalonych marginesach lewa-górna komórka ma rozkład hipergeometryczny. Dwustronne p-value liczymy jako sumę prawdopodobieństw tabel co najmniej tak mało prawdopodobnych jak obserwowana.

In [ ]:
M = obs.sum()
n = obs[0].sum()
N = obs[:, 0].sum()
start, end = hypergeom.support(M, n, N)
xs = np.arange(start, end + 1)
pmf = hypergeom.pmf(xs, M, n, N)
observed_prob = hypergeom.pmf(a, M, n, N)
two_sided_p = pmf[pmf <= observed_prob + 1e-12].sum()

fisher_manual = pd.DataFrame({'x_upper_left': xs, 'probability': pmf})
print('observed probability =', observed_prob)
print('two-sided p manual =', two_sided_p)
fisher_manual.tail(10)

## 3. Regresja logistyczna jako linia na log-odds

Model:

$$\log\left(\frac{p}{1-p}\right)=\beta_0+\beta_1x$$

Po odwróceniu dostajemy sigmoidę:

$$p=\frac{1}{1+e^{-(\beta_0+\beta_1x)}}$$

In [ ]:
etas = np.array([-3, -2, -1, 0, 1, 2, 3])
probs = expit(etas)
pd.DataFrame({'eta_log_odds': etas, 'probability': probs, 'odds': np.exp(etas)})

## 4. Demo StatQuest-style: heart disease i `sex`

Tabela:

| `hd` | F | M |
|---|---:|---:|
| Healthy | 71 | 89 |
| Unhealthy | 25 | 112 |

Dla `sexM=0` mamy kobiety, dla `sexM=1` mężczyzn. Współczynnik przy `sexM` powinien być log-odds ratio.

In [ ]:
rows = []
for sexM, healthy, unhealthy in [(0, 71, 25), (1, 89, 112)]:
    rows += [(sexM, 0)] * healthy
    rows += [(sexM, 1)] * unhealthy

df = pd.DataFrame(rows, columns=['sexM', 'unhealthy'])
X = np.column_stack([np.ones(len(df)), df['sexM'].to_numpy()])
y = df['unhealthy'].to_numpy()

def fit_logistic_newton(X, y, max_iter=100, tol=1e-10):
    beta = np.zeros(X.shape[1])
    history = []
    for iteration in range(max_iter):
        eta = X @ beta
        p = expit(eta)
        W = p * (1 - p)
        grad = X.T @ (y - p)
        H_pos = X.T @ (W[:, None] * X)
        step = np.linalg.solve(H_pos, grad)
        beta = beta + step
        ll = np.sum(y * (X @ beta) - np.logaddexp(0, X @ beta))
        history.append((iteration, ll, beta.copy(), np.max(np.abs(step))))
        if np.max(np.abs(step)) < tol:
            break
    return beta, history

beta, history = fit_logistic_newton(X, y)
manual_intercept = np.log(25/71)
manual_slope = np.log((112/89)/(25/71))

pd.DataFrame({
    'parameter': ['intercept', 'sexM'],
    'newton_beta': beta,
    'manual_beta': [manual_intercept, manual_slope],
    'exp_beta': np.exp(beta),
})

In [ ]:
pred = expit(np.array([[1,0], [1,1]]) @ beta)
pd.DataFrame({
    'group': ['F', 'M'],
    'predicted_P_unhealthy': pred,
    'empirical_P_unhealthy': [25/(25+71), 112/(112+89)]
})

## 5. Log-likelihood, McFadden pseudo-$R^2$ i likelihood-ratio test

$$\ell(\beta)=\sum_i[y_i\eta_i-\log(1+e^{\eta_i})]$$

$$R^2_{McFadden}=1-\frac{\ell_{full}}{\ell_{null}}$$

$$G^2=2(\ell_{full}-\ell_{null})$$

In [ ]:
def log_likelihood(X, y, beta):
    eta = X @ beta
    return np.sum(y * eta - np.logaddexp(0, eta))

ll_full = log_likelihood(X, y, beta)
X0 = np.ones((len(y), 1))
beta0 = np.array([np.log(y.mean() / (1-y.mean()))])
ll_null = log_likelihood(X0, y, beta0)

mcfadden = 1 - ll_full / ll_null
G2 = 2 * (ll_full - ll_null)
p_lrt = chi2.sf(G2, df=1)

pd.Series({
    'LL_null': ll_null,
    'LL_full': ll_full,
    'McFadden_R2': mcfadden,
    'G2_LRT': G2,
    'p_LRT': p_lrt,
    'null_deviance': -2*ll_null,
    'residual_deviance': -2*ll_full,
})

## 6. ROC/AUC ręcznie

AUC można policzyć jako prawdopodobieństwo, że losowy przykład pozytywny ma wyższy score niż losowy przykład negatywny.

In [ ]:
scores = np.array([0.95, 0.80, 0.70, 0.60, 0.55, 0.40, 0.35, 0.20])
y_true = np.array([1,    1,    0,    1,    0,    0,    1,    0])

thresholds = np.r_[np.inf, np.sort(scores)[::-1], -np.inf]
roc_rows = []
for t in thresholds:
    y_pred = (scores >= t).astype(int)
    TP = np.sum((y_true == 1) & (y_pred == 1))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    TPR = TP / (TP + FN)
    FPR = FP / (FP + TN)
    roc_rows.append((t, TP, FP, TN, FN, TPR, FPR))
roc_df = pd.DataFrame(roc_rows, columns=['threshold', 'TP', 'FP', 'TN', 'FN', 'TPR', 'FPR'])
roc_df

In [ ]:
pos_scores = scores[y_true == 1]
neg_scores = scores[y_true == 0]
count = 0
for ps in pos_scores:
    for ns in neg_scores:
        if ps > ns:
            count += 1
        elif ps == ns:
            count += 0.5
auc_pairwise = count / (len(pos_scores) * len(neg_scores))
auc_pairwise

## 7. Regularizacja — szkielety `sklearn`

W `sklearn` parametr `C` jest odwrotnością siły regularyzacji: mniejsze `C` oznacza silniejszą karę.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

model_l2 = make_pipeline(StandardScaler(), LogisticRegression(penalty='l2', C=1.0, solver='lbfgs', max_iter=1000))
model_l1 = make_pipeline(StandardScaler(), LogisticRegression(penalty='l1', C=1.0, solver='saga', max_iter=5000))
model_en = make_pipeline(StandardScaler(), LogisticRegression(penalty='elasticnet', C=1.0, l1_ratio=0.5, solver='saga', max_iter=5000))

print(model_l2)
print(model_l1)
print(model_en)

## 8. Mini-checklista prowadzącego

- Zacznij od $p$, odds i log-odds.
- Dopiero potem pokaż $logit(p)=X\beta$.
- Następnie odwróć logit do sigmoidy.
- Współczynników nie interpretuj jako zmiany probability. Interpretuj je jako zmianę log-odds albo po eksponencjacji jako odds ratio.
- Oddziel score/probability od progu klasyfikacji.
- Przy testach pokaż różnicę: Fisher dokładny, chi-square asymptotyczny dla tabel, Wald asymptotyczny dla współczynnika/log(OR).